## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-03-21-54-16 +0000,nypost,California candidate’s Election Day fits ranke...,https://nypost.com/2026/11/03/us-news/californ...
1,2026-11-03-21-52-37 +0000,nyt,Trump Says U.S. to End $4 Billion in Aid to Is...,https://www.nytimes.com/2026/11/03/us/politics...
2,2026-11-03-21-52-00 +0000,wsj,"Booking Profit, Sales Fall as Global Conflicts...",https://www.wsj.com/business/earnings/booking-...
3,2026-11-03-21-52-00 +0000,wsj,Learjet Wreckage Found Miles Away From Crash S...,https://www.wsj.com/us-news/learjet-wreckage-f...
4,2026-11-03-21-51-04 +0000,nyt,Democrats rise in polls of New Hampshire and M...,https://www.nytimes.com/live/2026/11/03/us/mid...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2448/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
2,election,47
13,trump,42
3,day,28
158,midterm,23
66,ai,17


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
270,2026-11-03-10-56-00 +0000,wsj,Trump surprises Republicans with an election-e...,https://www.wsj.com/politics/elections/control...,183
63,2026-11-03-20-32-42 +0000,chitri,Lake County Election Day turnout strong and st...,https://www.chicagotribune.com/2026/11/03/lake...,156
237,2026-11-03-11-57-00 +0000,wsj,"Follow the WSJ’s Election Day live coverage, a...",https://www.wsj.com/politics/elections/your-ho...,146
121,2026-11-03-18-11-21 +0000,nypost,Alaska Senate candidate ‘Decoy’ Dan Sullivan m...,https://nypost.com/2026/11/03/us-news/alaska-s...,146
147,2026-11-03-16-53-19 +0000,nyt,Trump Makes Last Pitch to Voters on Election Day,https://www.nytimes.com/2026/11/03/us/trump-mi...,136


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
270,183,2026-11-03-10-56-00 +0000,wsj,Trump surprises Republicans with an election-e...,https://www.wsj.com/politics/elections/control...
331,115,2026-11-03-02-00-00 +0000,wsj,Tuesday’s midterm elections will offer clues t...,https://www.wsj.com/politics/elections/what-th...
237,82,2026-11-03-11-57-00 +0000,wsj,"Follow the WSJ’s Election Day live coverage, a...",https://www.wsj.com/politics/elections/your-ho...
371,66,2026-11-02-22-40-00 +0000,wsj,Midterm candidates across the country are maki...,https://www.wsj.com/politics/policy/midterm-ca...
78,51,2026-11-03-20-08-11 +0000,nypost,Second California student hurt in deadly Spain...,https://nypost.com/2026/11/03/us-news/second-c...
262,49,2026-11-03-11-00-00 +0000,nypost,Campaign to keep Ilhan Omar’s secrets safe inc...,https://nypost.com/2026/11/03/us-news/campaign...
266,45,2026-11-03-11-00-00 +0000,latimes,Hiltzik: Can AI bots become conscious? Why Mic...,https://www.latimes.com/business/story/2026-11...
65,44,2026-11-03-20-29-00 +0000,wsj,Trump Says U.S. Plans to Cut Off Military Aid ...,https://www.wsj.com/world/middle-east/trump-sa...
251,43,2026-11-03-11-05-43 +0000,chitri,‘Absolutely astonishing’: Pastry fans line up ...,https://www.chicagotribune.com/2026/11/03/trit...
309,42,2026-11-03-07-53-38 +0000,bbc,Major test for Australia's social media ban be...,https://www.bbc.co.uk/news/articles/ck79py02p1...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
